In [6]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox, Layout
from IPython.display import display

# ==========================================
# 1. 模拟生成流数据 (包含真实心跳与噪声)
# ==========================================
FS = 100                # 采样频率 100Hz
TOTAL_TIME = 15.0       # 模拟总时长 15 秒

# 生成时间轴
t_all = np.arange(0, TOTAL_TIME, 1.0 / FS)

# 模拟真实的“无噪声”PPG波形 (主峰 + 重搏波)
# 1.2Hz 代表心率约 72 bpm (真实周期为 1 / 1.2 = 0.83333s)
clean_ppg = 1000 * np.sin(2 * np.pi * 1.2 * t_all) + 300 * np.cos(2 * np.pi * 2.4 * t_all - 1.0)

# 生成固定的随机噪声底底，防止滑块拖动时噪声跳变导致视觉干扰
np.random.seed(42)
base_noise = np.random.normal(0, 1.0, len(t_all))

def compute_fourier_basis(t_array: np.ndarray, q: int, period: float) -> np.ndarray:
    """计算傅里叶正交基矩阵 Phi"""
    N = len(t_array)
    Phi = np.zeros((N, q))
    phase = (t_array % period) / period
    
    Phi[:, 0] = 1.0  # 直流分量
    for k in range(1, (q + 1) // 2):
        Phi[:, 2*k - 1] = np.sqrt(2) * np.cos(2 * np.pi * k * phase)
        if 2*k < q:
            Phi[:, 2*k] = np.sqrt(2) * np.sin(2 * np.pi * k * phase)
    return Phi

# ==========================================
# 2. 核心交互可视化函数
# ==========================================
def interactive_ppg_filter(
    current_time=14.5,  # 当前时间窗口 
    basis_period=0.833, # 基准周期 (默认拨至完美共振点)
    q=5,                # 基函数数量 (真实信号只有两个频率，q=5为理论完美自由度)
    lam=0.98,           # 滑动遗忘因子
    use_cumulative=True,# 【新增】：是否使用全局累加模式（等价于全量数据OLS最小二乘）
    rho=0.00,           # 岭回归惩罚系数 
    noise_level=230.0   # 原始数据的噪声强度
):
    y_noisy = clean_ppg + base_noise * noise_level
    
    BATCH_SIZE = 100  # 1秒微批次
    current_idx = int(current_time * FS)
    
    G_state = np.zeros(q)
    H_state = np.zeros((q, q))
    step = 0
    
    # 模拟数据不断流入，更新状态
    for i in range(0, current_idx, BATCH_SIZE):
        end_idx = min(i + BATCH_SIZE, current_idx)
        if end_idx - i == 0: break
            
        t_batch = t_all[i:end_idx]
        y_batch = y_noisy[i:end_idx]
        
        Phi = compute_fourier_basis(t_batch, q, basis_period)
        G_batch = np.dot(Phi.T, y_batch) / len(t_batch)
        H_batch = np.dot(Phi.T, Phi) / len(t_batch)
        
        # 【核心逻辑分流】
        if use_cumulative:
            # 模式 A：全局累加模式 (非流式局部窗口)
            # 权重随数据量动态调整，完全不遗忘历史。等价于拥有全部历史数据的完美数学期望。
            lam_dynamic = step / (step + 1.0) 
            G_state = lam_dynamic * G_state + (1 - lam_dynamic) * G_batch
            H_state = lam_dynamic * H_state + (1 - lam_dynamic) * H_batch
        else:
            # 模式 B：指数滑动窗口模式 (真实可穿戴设备使用，捕捉心率变化)
            G_state = lam * G_state + (1 - lam) * G_batch
            H_state = lam * H_state + (1 - lam) * H_batch
            
        step += 1

    # --- 偏差修正与求解 ---
    if not use_cumulative and step > 0:
        bias_correction = 1.0 - lam**step
        G_corr = G_state / bias_correction
        H_corr = H_state / bias_correction
    else:
        # 全局累加模式天生无偏差
        G_corr = G_state
        if step == 0:
            H_corr = np.eye(q)
        else:
            H_corr = H_state

    # 求解系数
    a_hat = np.linalg.solve(H_corr + rho * np.eye(q), G_corr)
    
    plot_window_seconds = 3.0
    start_idx = max(0, current_idx - int(plot_window_seconds * FS))
    
    t_plot = t_all[start_idx:current_idx]
    y_raw_plot = y_noisy[start_idx:current_idx]
    y_clean_plot = clean_ppg[start_idx:current_idx]
    
    Phi_plot = compute_fourier_basis(t_plot, q, basis_period)
    y_recon = np.dot(Phi_plot, a_hat)

    # --- 绘制图表 ---
    plt.figure(figsize=(12, 6))
    
    plt.plot(t_plot, y_raw_plot, color='lightgray', label='Sensor Raw (Noisy)', alpha=0.7)
    # 将蓝线加粗，方便观察它是否被红线完美覆盖
    plt.plot(t_plot, y_clean_plot, color='blue', linestyle='--', label='Ground Truth', linewidth=3.0)
    plt.plot(t_plot, y_recon, color='red', label=f'Algorithm Output (q={q})', linewidth=2.0)
    
    mode_text = "Cumulative Mode (Global OLS)" if use_cumulative else f"Sliding Window (lam={lam:.3f})"
    plt.title(f"One-pass Estimator Real-time Reconstruction (Time: {current_time:.1f}s)\n" 
              f"[{mode_text}] Period={basis_period:.3f}s, q={q}, rho={rho:.2f}", fontsize=14)
    plt.xlabel("Time (seconds)", fontsize=12)
    plt.ylabel("PPG Intensity", fontsize=12)
    plt.xlim(max(0, current_time - plot_window_seconds), current_time)
    
    plt.ylim(-1500, 1500)
    plt.legend(loc='upper right', fontsize=10)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. 绑定 UI 滑块
# ==========================================
print("请在 Jupyter 环境下运行，体验【理论最优解】完美重合：")
interact(
    interactive_ppg_filter,
    use_cumulative=Checkbox(value=True, description='使用全局累加(验证理论最优解)', layout=Layout(width='500px')),
    current_time=FloatSlider(value=14.5, min=3.0, max=15.0, step=0.5, description='时间窗滑动:', layout=Layout(width='500px')),
    basis_period=FloatSlider(value=0.833, min=0.5, max=3.0, step=0.001, description='基准周期(s):', layout=Layout(width='500px')),
    q=IntSlider(value=5, min=1, max=35, step=2, description='基函数 (q):', layout=Layout(width='500px')),
    lam=FloatSlider(value=0.98, min=0.80, max=0.999, step=0.005, description='遗忘因子(滑动模式用):', layout=Layout(width='500px')),
    rho=FloatSlider(value=0.00, min=0.0, max=0.2, step=0.01, description='惩罚系数(ρ):', layout=Layout(width='500px')),
    noise_level=FloatSlider(value=230.0, min=0.0, max=500.0, step=10.0, description='噪声强度:', layout=Layout(width='500px'))
);

请在 Jupyter 环境下运行，体验【理论最优解】完美重合：


interactive(children=(FloatSlider(value=14.5, description='时间窗滑动:', layout=Layout(width='500px'), max=15.0, mi…

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [9]:
df = pd.read_csv("D:/Workspace/5_Data/bupt_ring_selftest/wsx/报告/green/parsed_results/ring.csv")
raw_y = -1. * df["green1"].values

In [4]:
print(df.head(5).to_string())

      green1     green2             Datetime  motion  motion_300
0  8715406.0  8502910.0  2026-06-10 11:31:23    2091           0
1  8716251.0  8503157.0  2026-06-10 11:31:23    2078           0
2  8715944.0  8503201.0  2026-06-10 11:31:23    2011           0
3  8715814.0  8503174.0  2026-06-10 11:31:23    1995           0
4  8715142.0  8503446.0  2026-06-10 11:31:23    1990           0
